# Move legacy processed world videos into numbered recording directories

This notebook migrates processed world videos from the legacy layout:

```text
FLIC_<subject>/<activity>/GKA/W.avi
```

to the layout expected by the current preprocessing pipeline:

```text
FLIC_<subject>/<activity>/GKA/<recording_number>/W.avi
```

The recording numbers are read from the matching raw-data `GKA` directory. If an activity has multiple numbered raw recordings, the default policy selects the highest number because the legacy `transfer_light_logger_recordings(...)` implementation retained the highest-numbered attempt. You can override any ambiguous choice in `RECORDING_NUMBER_OVERRIDES`.

The notebook starts in preview-only mode and never overwrites a destination video. If the destination `W.avi` already exists, that legacy source is simply skipped. Review the plan carefully before setting `APPLY_MOVES = True`.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import re
import shutil

RAW_DATASET_DIR = Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026")
PROCESSING_DATASET_DIR = Path("/Volumes/FLIC_processing/NEWscriptedIndoorOutdoorVideos2026")

# Leave False for the first run. Change to True only after reviewing the plan.
APPLY_MOVES = False

# Optional filters. Empty sets mean all subjects and all activities.
SUBJECTS_TO_PROCESS: set[int] = set()
SUBJECTS_TO_SKIP: set[int] = set()
ACTIVITIES_TO_PROCESS: set[str] = set()
ACTIVITIES_TO_SKIP: set[str] = set()

# Use (subject_number, activity_name): recording_number for exceptions.
# Example: RECORDING_NUMBER_OVERRIDES[(2001, "walkOutdoor")] = 2
RECORDING_NUMBER_OVERRIDES: dict[tuple[int, str], int] = {}

In [ ]:
@dataclass(frozen=True)
class VideoMove:
    subject_number: int
    activity_name: str
    source: Path
    destination: Path
    available_recording_numbers: tuple[int, ...]
    selection_reason: str


def is_selected(item, items_to_process, items_to_skip) -> bool:
    return item in items_to_process if items_to_process else item not in items_to_skip


def numbered_raw_recordings(raw_gka_dir: Path) -> tuple[int, ...]:
    if not raw_gka_dir.is_dir():
        return ()
    return tuple(sorted(
        int(path.name)
        for path in raw_gka_dir.iterdir()
        if path.is_dir() and path.name.isdigit()
    ))


def build_video_move_plan(raw_root: Path, processing_root: Path):
    raw_root = raw_root.expanduser().resolve()
    processing_root = processing_root.expanduser().resolve()
    if not raw_root.is_dir():
        raise NotADirectoryError(f"Raw dataset root is not a directory: {raw_root}")
    if not processing_root.is_dir():
        raise NotADirectoryError(f"Processing dataset root is not a directory: {processing_root}")

    moves: list[VideoMove] = []
    conflicts: list[str] = []
    ignored_overrides = set(RECORDING_NUMBER_OVERRIDES)

    subject_dirs = sorted(
        (path for path in processing_root.iterdir()
         if path.is_dir() and re.fullmatch(r"FLIC_\d+", path.name)),
        key=lambda path: int(path.name.removeprefix("FLIC_")),
    )

    for subject_dir in subject_dirs:
        subject_number = int(subject_dir.name.removeprefix("FLIC_"))
        if not is_selected(subject_number, SUBJECTS_TO_PROCESS, SUBJECTS_TO_SKIP):
            continue

        activity_dirs = sorted((p for p in subject_dir.iterdir() if p.is_dir()), key=lambda p: p.name)
        for activity_dir in activity_dirs:
            activity_name = activity_dir.name
            if not is_selected(activity_name, ACTIVITIES_TO_PROCESS, ACTIVITIES_TO_SKIP):
                continue

            source = activity_dir / "GKA" / "W.avi"
            if not source.is_file():
                continue

            raw_gka_dir = raw_root / subject_dir.name / activity_name / "GKA"
            recording_numbers = numbered_raw_recordings(raw_gka_dir)
            if not recording_numbers:
                conflicts.append(
                    f"No numbered raw recording directories found for {source}; checked {raw_gka_dir}"
                )
                continue

            override_key = (subject_number, activity_name)
            if override_key in RECORDING_NUMBER_OVERRIDES:
                recording_number = RECORDING_NUMBER_OVERRIDES[override_key]
                ignored_overrides.discard(override_key)
                if recording_number not in recording_numbers:
                    conflicts.append(
                        f"Override {override_key} -> {recording_number} is not present in raw recordings "
                        f"{recording_numbers}: {raw_gka_dir}"
                    )
                    continue
                selection_reason = "explicit override"
            elif len(recording_numbers) == 1:
                recording_number = recording_numbers[0]
                selection_reason = "only raw recording"
            else:
                recording_number = max(recording_numbers)
                selection_reason = "highest raw recording (legacy transfer behavior)"

            destination = source.parent / str(recording_number) / source.name
            if destination.exists():
                # The numbered video has already been generated or migrated.
                # Leave both files untouched and continue with other activities.
                continue

            moves.append(VideoMove(
                subject_number=subject_number,
                activity_name=activity_name,
                source=source,
                destination=destination,
                available_recording_numbers=recording_numbers,
                selection_reason=selection_reason,
            ))

    for override_key in sorted(ignored_overrides):
        conflicts.append(
            f"Unused override {override_key}: no matching selected legacy GKA/W.avi was found"
        )

    return moves, conflicts


def print_video_move_plan(moves: list[VideoMove], conflicts: list[str]) -> None:
    print(f"Legacy videos ready to move: {len(moves)}")
    for move in moves:
        print(f"\n{move.source}")
        print(f"  -> {move.destination}")
        print(f"  raw recording numbers: {move.available_recording_numbers}")
        print(f"  selected by: {move.selection_reason}")
    if conflicts:
        print(f"\nCONFLICTS ({len(conflicts)}; these videos will not be moved):")
        for conflict in conflicts:
            print(f"  - {conflict}")

## Preview the move plan

Run this cell and inspect every multi-recording choice. Add entries to `RECORDING_NUMBER_OVERRIDES` in the configuration cell if any legacy video belongs to a recording other than the highest-numbered attempt.

In [ ]:
moves, conflicts = build_video_move_plan(RAW_DATASET_DIR, PROCESSING_DATASET_DIR)
print_video_move_plan(moves, conflicts)

## Apply the moves

This cell rebuilds the plan immediately before execution. Existing destination videos are skipped during planning. It aborts without moving anything only for true mapping conflicts. A destination file is checked again immediately before each move and is never overwritten.

In [ ]:
def apply_video_moves(moves: list[VideoMove], apply_moves: bool = False) -> None:
    if not apply_moves:
        print("PREVIEW ONLY: no videos were moved. Set APPLY_MOVES = True and rerun this cell to apply.")
        return

    for move in moves:
        if not move.source.is_file():
            raise FileNotFoundError(f"Planned source disappeared before it could be moved: {move.source}")
        if move.destination.exists():
            raise FileExistsError(f"Refusing to overwrite destination: {move.destination}")
        move.destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(move.source), str(move.destination))
        print(f"Moved: {move.source} -> {move.destination}")


# Rebuild immediately before execution so a stale preview is never applied.
moves, conflicts = build_video_move_plan(RAW_DATASET_DIR, PROCESSING_DATASET_DIR)
if conflicts:
    print_video_move_plan(moves, conflicts)
    raise RuntimeError("Resolve the conflicts printed above before applying any moves.")
apply_video_moves(moves, apply_moves=APPLY_MOVES)

## Verify the migration

After applying, this reports any remaining selected legacy videos. It also counts processed videos already stored in numbered recording directories.

In [ ]:
def verify_video_layout(processing_root: Path) -> None:
    processing_root = processing_root.expanduser().resolve()
    remaining_legacy: list[Path] = []
    numbered_videos: list[Path] = []

    for subject_dir in processing_root.iterdir():
        if not subject_dir.is_dir() or not re.fullmatch(r"FLIC_\d+", subject_dir.name):
            continue
        subject_number = int(subject_dir.name.removeprefix("FLIC_"))
        if not is_selected(subject_number, SUBJECTS_TO_PROCESS, SUBJECTS_TO_SKIP):
            continue
        for activity_dir in (path for path in subject_dir.iterdir() if path.is_dir()):
            if not is_selected(activity_dir.name, ACTIVITIES_TO_PROCESS, ACTIVITIES_TO_SKIP):
                continue
            gka_dir = activity_dir / "GKA"
            legacy_video = gka_dir / "W.avi"
            if legacy_video.is_file():
                remaining_legacy.append(legacy_video)
            if gka_dir.is_dir():
                numbered_videos.extend(
                    path / "W.avi"
                    for path in gka_dir.iterdir()
                    if path.is_dir() and path.name.isdigit() and (path / "W.avi").is_file()
                )

    print(f"Numbered processed world videos: {len(numbered_videos)}")
    print(f"Remaining legacy GKA/W.avi files: {len(remaining_legacy)}")
    for path in sorted(remaining_legacy):
        print(f"  {path}")


verify_video_layout(PROCESSING_DATASET_DIR)